In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 · Bronze ingestion (Auto Loader)
# MAGIC
# MAGIC Ingests the three raw feeds from the landing Volume into Bronze Delta tables.
# MAGIC
# MAGIC | Feed | Format | Bronze table |
# MAGIC |---|---|---|
# MAGIC | `landing/logs/` | JSON lines | `bronze.app_logs_raw` |
# MAGIC | `landing/incidents/` | CSV | `bronze.incident_updates_raw` |
# MAGIC | `landing/deployments/` | CSV | `bronze.deployments_raw` |
# MAGIC
# MAGIC **Bronze rules:** keep the data as it arrived (all columns as strings, no dedupe, no filtering),
# MAGIC and add lineage columns so every row can be traced back to its source file.
# MAGIC
# MAGIC Runs with `trigger(availableNow=True)`: processes only files not seen before, then stops.
# MAGIC (Databricks Free Edition runs on serverless compute, which supports this trigger, not continuous streaming.)

# COMMAND ----------

from pyspark.sql import functions as F

CATALOG = "incident_analytics"
LANDING = f"/Volumes/{CATALOG}/raw/landing"
STATE = f"/Volumes/{CATALOG}/raw/checkpoints"   # schema locations + checkpoints live here

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.raw.checkpoints")

# COMMAND ----------

def ingest_to_bronze(source: str, file_format: str, table: str, extra_options: dict | None = None):
    """Incrementally load new files from landing/<source>/ into bronze.<table>."""
    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", f"{STATE}/{source}/_schema")  # where the inferred schema is tracked
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")          # new columns -> stream stops once, then picks them up
        .option("rescuedDataColumn", "_rescued_data")                       # values that don't fit the schema land here
    )
    for key, value in (extra_options or {}).items():
        reader = reader.option(key, value)

    df = (
        reader.load(f"{LANDING}/{source}/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias("_file_modified_at"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )

    query = (
        df.writeStream
        .option("checkpointLocation", f"{STATE}/{source}/_checkpoint")  # remembers which files are done
        .option("mergeSchema", "true")                                   # let the Delta table accept new columns
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.bronze.{table}")
    )
    query.awaitTermination()
    print(f"{source:12s} -> bronze.{table}: done")

# COMMAND ----------

ingest_to_bronze("logs", "json", "app_logs_raw")
ingest_to_bronze("incidents", "csv", "incident_updates_raw", {"header": "true"})
ingest_to_bronze("deployments", "csv", "deployments_raw", {"header": "true"})

# COMMAND ----------

# MAGIC %md
# MAGIC ## Checks — look at what landed

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Rows per source file (one row per batch file you've ingested)
# MAGIC SELECT _source_file, COUNT(*) AS row_count, MIN(_ingested_at) AS ingested_at
# MAGIC FROM incident_analytics.bronze.app_logs_raw
# MAGIC GROUP BY _source_file
# MAGIC ORDER BY _source_file;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Data quality problems Silver will have to fix
# MAGIC SELECT
# MAGIC   COUNT(*)                                       AS total_rows,
# MAGIC   COUNT(*) - COUNT(DISTINCT event_id)            AS duplicate_rows,
# MAGIC   SUM(CASE WHEN event_id IS NULL THEN 1 END)     AS rows_without_event_id,
# MAGIC   SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 END) AS rows_with_rescued_data
# MAGIC FROM incident_analytics.bronze.app_logs_raw;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- How messy are levels and timestamps?
# MAGIC SELECT level, COUNT(*) AS n FROM incident_analytics.bronze.app_logs_raw GROUP BY level ORDER BY n DESC;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT timestamp, _source_file
# MAGIC FROM incident_analytics.bronze.app_logs_raw
# MAGIC WHERE timestamp NOT LIKE '%T%Z'
# MAGIC LIMIT 20;
